# Discover algorithms with esa_apex_toolbox

Discover algorithms in the [ESA APEx catalogue](https://algorithm-catalogue.apex.esa.int/), filter their metadata, and inspect an openEO user-defined process (UDP). The toolbox discovers services; it does not itself process satellite imagery.

Use a Python 3.8+ kernel and run this notebook from a clone of `ESA-APEx/apex_algorithms`. Internet access is required for the GitHub examples. No credentials are needed and no processing jobs are submitted. Network errors propagate; public GitHub API rate limits apply.

## Install the toolbox

Install the current checkout in editable mode so its discovery and filtering APIs are available. The package distribution is named `esa-apex-algorithms`, while its Python import is `esa_apex_toolbox`. Open the notebook from the repository root or the `examples` directory. Restart the kernel after installation if it already imported an older version.

In [1]:
from pathlib import Path

project_root = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / 'pyproject.toml').is_file()
        and (path / 'src' / 'esa_apex_toolbox').is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError('Open this notebook from the apex_algorithms checkout.')
print(project_root)

/Users/bramjanssen/projects/apex/apex_algorithms


In [2]:
%pip install -e "{project_root}[notebook]"

Obtaining file:///Users/bramjanssen/projects/apex/apex_algorithms
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for esa-apex-algorithms (pyproject.toml) ... done
  Created wheel for esa-apex-algorithms: filename=esa_apex_algorithms-0.0.1-py3-none-any.whl size=11064 sha256=9e6091606a46253124312f1c18a392bda64f4c0ed6612e71c6ead18e27ba71ee
  Stored in directory: /private/var/folders/50/09_2zmx12zj6ks4fdl4y9wgc0000gn/T/pip-ephem-wheel-cache-r0aekixx/wheels/7b/1c/2c/eb3e740b89a0298b13dc54e2f85f4e3b953caac511a15efdfb
Successfully built esa-apex-algorithms
  Attempting uninstall: esa-apex-algorithms
    Found existing installation: esa-apex-algorithms 0.0.1
    Uninstalling esa-apex-algorithms-0.0.1:
      Successfully uninstalled esa-apex-algorithms-0.0.1
Note: you may ne

## Connect and list algorithm IDs

The repository reads public GitHub directory listings. `branch` selects the catalogue revision, so the metadata below comes from GitHub, not uncommitted local records. Listing IDs alone does not download every record.

### Why a GitHub token can help

A token is optional for this public catalogue. GitHub normally allows **60 unauthenticated API requests per hour per IP address**, shared with other users and tools on that IP. Repeated notebook runs or creating new repository instances can exhaust this allowance and cause `403 rate limit exceeded`. A personal access token normally raises the primary limit to **5,000 requests per hour per user**, shared with your other authenticated applications. It does not remove all rate limits or provide access to an openEO backend. Reuse `repo` to benefit from its cache. See [GitHub's rate-limit documentation](https://docs.github.com/en/rest/using-the-rest-api/rate-limits-for-the-rest-api).

### Create a token

1. Sign in to GitHub and open **Settings > Developer settings > Personal access tokens > Fine-grained tokens > Generate new token**, or go directly to the [token creation page](https://github.com/settings/personal-access-tokens/new).
2. Give it a name such as `APEx catalogue` and choose a short expiration appropriate for your work.
3. Keep your own account as **Resource owner** and choose **Public repositories** under **Repository access**. Public read-only access is included; membership in `ESA-APEx` is not required.
4. Leave additional repository and account permissions unset. This public catalogue needs neither private-repository access nor write permissions.
5. Select **Generate token** and keep the value in a password manager or secret store; GitHub only shows it when it is created.

See [GitHub's fine-grained token instructions](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens#creating-a-fine-grained-personal-access-token) for current settings.

### Use it in this notebook

The next cell reads `GITHUB_TOKEN` or `GH_TOKEN` from the kernel's environment and passes it as `token=`. If neither is set, enter the token in the hidden `getpass` prompt. Press Enter for unauthenticated access if quota remains. Never paste a token into notebook source, print it, or share it in chat. Revoke and replace an exposed token, and replace it when it expires.

If rate-limited, wait until the reset time in GitHub's `X-RateLimit-Reset` response header, or follow `Retry-After` when present. Do not repeatedly rerun failing requests. `401 Bad credentials` usually indicates an invalid, expired, or revoked token. Restart the kernel after upgrading the toolbox so the `token` argument is available.

The [toolbox guide](https://esa-apex.github.io/apex_algorithms/toolbox.html#github-authentication) has more details.

### Browse and search the catalogue

The next cell uses `list_algorithms(details=True)` to load the catalogue, then `display(algorithms.widget())` to show a search field above the expandable table. Search matches the algorithm title/name, ID, provider, platforms, and keywords. Matching ignores case and accepts partial words. Every space-separated term must match somewhere in those fields: for example, `vito sentinel` combines a provider and a keyword.

Results update while you type. Clear the field to restore the full list. Searching only filters the already-loaded algorithms: it does not make more network requests or change the original list. Click an algorithm title to inspect its description and service links.

The notebook installation cell includes the optional `ipywidgets` dependency. Keep the notebook kernel running for interactive search. Use `display(algorithms)` instead for the static, dependency-free table, or `display(algorithm)` for a single record. Restart the kernel after upgrading the toolbox.

In [3]:
import os
from getpass import getpass
from importlib.metadata import version

from IPython.display import display

from esa_apex_toolbox.algorithms import Algorithm, GithubAlgorithmRepository

github_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
if not github_token:
    github_token = getpass('GitHub token (optional; Enter for unauthenticated access): ') or None

print('Toolbox version:', version('esa-apex-algorithms'))
repo = GithubAlgorithmRepository(
    owner='ESA-APEx',
    repo='apex_algorithms',
    folder='algorithm_catalog',
    branch='main',
    token=github_token,
)
algorithms = repo.list_algorithms(details=True)
display(algorithms.widget())

Toolbox version: 0.0.1


## Inspect one algorithm

Pass a folder ID from `list_algorithms()` to `get_algorithm()`. Here we inspect the Sentinel-2 maximum-NDVI composite. The provider comes from a catalogue link; `organization` is the principal investigator's organization and can differ. Missing metadata may be `None` or an empty list.

In [4]:
algorithm = repo.get_algorithm('max_ndvi_composite')
display(algorithm)

Algorithm(id='max_ndvi_composite', title='Max NDVI Composite based on Sentinel-2 data', description='A compositing algorithm for Sentinel-2 L2A data, ranking observations by their maximum NDVI. The cost depends on both the area and number of input observations.', udp_link=UdpLink(href='https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/main/algorithm_catalog/vito/max_ndvi_composite/openeo_udp/max_ndvi_composite.json', title='openEO Process Definition'), service_links=[ServiceLink(href='https://openeofed.dataspace.copernicus.eu', title='CDSE openEO federation')], license=None, organization='VITO', provider='VITO', platforms=['CDSE openEO federation'], keywords=['Vegetation', 'Normalized Difference Vegetation Index (NDVI)', 'Sentinel-2'])

## Discover filter values

These helpers return sorted, unique provider titles, platform titles, and keywords. The first call downloads catalogue records in parallel and can take a little time. Listings and successfully parsed records are cached on `repo`; reuse it to avoid repeat downloads, or create a new instance to refresh. Provider and platform titles are not folder names.

In [5]:
providers = repo.list_providers()
platforms = repo.list_platforms()
keywords = repo.list_keywords()
print('Providers:', providers)
print('Platforms:', platforms)
print('Keywords:', keywords)

Providers: ['APEx', 'Brockmann Consult GmbH', 'CS GROUP - ROMANIA', 'DHI', 'DLR', 'David Frantz', 'Development Seed', 'Earth RayView Ltd', 'EarthDaily', 'Eurac Research', 'European Space Agency', 'GISAT', 'OBSGESSION', 'TERRAVISION', 'Terradue', 'VITO', 'Wageningen University & Research']
Platforms: ['CDSE openEO Federation processing backend', 'CDSE openEO federation', 'Copernicus Data Space openEO Backend', 'EOxHub Workspaces', 'Terrascope', 'TiTiler openEO', 'Tool Augmentation by user enhancements and Orchestration (TAO)', 'VITO']
Keywords: ['Agriculture', 'Brightness', 'Change Detection Services', 'Climate', 'Data Analysis and Visualization', 'Deforestation', 'Digital Elevation/Terrain Model (DEM)', 'ECMWF ERA5', 'Energy Production/Use', 'Floods', 'Forests', 'Fraction of Absorbed Photosynthetic Active Radiation (fAPAR)', 'Fraction of Vegetation Coverage (fCOVER)', 'Geology', 'Hydrology', 'Land Surface Temperature', 'Land Use/Land Cover Change', 'Land Use/Land Cover Classification',

## Filter algorithms

Filters are case-insensitive **exact** matches and combine with AND. A provider or platform list matches **any** supplied title; a keyword list requires **all** supplied keywords. Empty provider/platform lists match nothing, while an empty keyword list imposes no restriction.

Change these example values using the discovered titles above. Without `details=True`, the result contains folder IDs; with it, the result contains `Algorithm` objects. No matches is a valid result.

In [6]:
matches = repo.list_algorithms(
    provider=['VITO'],
    platform='CDSE openEO federation',
    details=True,
)
display(matches.widget())

## Inspect an openEO process definition

Not every algorithm has a UDP: some publish CWL application packages instead. When a UDP is available, download its JSON and inspect the input parameters and graph size. This does **not** execute the process. Actual execution needs an openEO client, a supported backend, authentication, suitable parameter values, and potentially processing credits.

In [7]:
import requests

process_definition = None
if algorithm.udp_link is None:
    print('This algorithm does not advertise an openEO process definition.')
else:
    print('Process URL:', algorithm.udp_link.href)
    response = requests.get(algorithm.udp_link.href, timeout=30)
    response.raise_for_status()
    process_definition = response.json()
    print('Process ID:', process_definition.get('id'))
    for parameter in process_definition.get('parameters', []):
        print(parameter['name'], '|', parameter.get('description', ''))
    print('Graph nodes:', len(process_definition.get('process_graph', {})))

Process URL: https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/main/algorithm_catalog/vito/max_ndvi_composite/openeo_udp/max_ndvi_composite.json
Process ID: max_ndvi_composite
spatial_extent | Limits the data to process to the specified bounding box or polygons.

For raster data, the process loads the pixel into the data cube if the point
at the pixel center intersects with the bounding box or any of the polygons
(as defined in the Simple Features standard by the OGC).

For vector data, the process loads the geometry into the data cube if the geometry
is fully within the bounding box or any of the polygons (as defined in the
Simple Features standard by the OGC). Empty geometries may only be in the
data cube if no spatial extent has been provided.

Empty geometries are ignored.

Set this parameter to null to set no limit for the spatial extent.
temporal_extent | Temporal extent specified as two-element array with start and end date/date-time.
max_cloud_cover | The maximum cloud

## Load a local catalogue record

`Algorithm.from_ogc_api_record()` also accepts a dictionary, a JSON string, or an HTTP(S) URL. Reading a local file works offline and lets you inspect uncommitted catalogue metadata.

In [8]:
record_path = (
    project_root / 'algorithm_catalog' / 'vito' / 'max_ndvi_composite'
    / 'records' / 'max_ndvi_composite.json'
)
local_algorithm = Algorithm.from_ogc_api_record(record_path)
print(local_algorithm.id, '|', local_algorithm.title)

max_ndvi_composite | Max NDVI Composite based on Sentinel-2 data


## Next steps

See the [repository README](../README.md) for API details and browse the [APEx Algorithm Catalogue](https://algorithm-catalogue.apex.esa.int/) for service documentation. A service link identifies where an algorithm can run; discovering that link does not start a processing job.